# Fundamentos de redes neuronales — Práctica: el perceptrón

Implementación desde cero del **perceptrón** y su entrenamiento con la **compuerta lógica AND**,
siguiendo la teoría del módulo (`../teoria/`).

Contenido:

1. Los datos: la tabla de verdad de AND
2. Las piezas del perceptrón: suma ponderada y función de activación
3. La regla de aprendizaje (regla delta)
4. Entrenamiento paso a paso
5. Reproducción del ejemplo manual de las slides
6. Visualización de la frontera de decisión
7. La limitación: el problema del XOR
8. Implementación con scikit-learn


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)   # reproducibilidad

## 1. Los datos: tabla de verdad de la compuerta AND

| x1 | x2 | x1 AND x2 |
|----|----|-----------|
| 0  | 0  | 0         |
| 0  | 1  | 0         |
| 1  | 0  | 0         |
| 1  | 1  | 1         |

Solo devuelve 1 cuando **ambas** entradas son 1.


In [ ]:
entradas = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

# Compuerta logica AND
etiquetas = np.array([0, 0, 0, 1])

print("entradas:", entradas.shape, "| etiquetas:", etiquetas.shape)

In [ ]:
# Visualizar los datos
plt.scatter(entradas[:, 0], entradas[:, 1], c=etiquetas, cmap="cool", marker="o", s=150)
plt.title("Compuerta logica AND")
plt.xlabel("x1")
plt.ylabel("x2")
plt.grid(True)
plt.show()

Las cuatro combinaciones son **linealmente separables**: se puede trazar una recta que deje
el punto (1,1) de un lado y los otros tres del otro. Esa es exactamente la condición que el
perceptrón necesita para converger.

## 2. Las piezas del perceptrón

**Suma ponderada:**

```
z = w1*x1 + w2*x2 + theta
```

**Función de activación (escalón):**

```
phi(z) = 1  si z >= 0
phi(z) = 0  si z <  0
```

donde `theta` es el umbral o sesgo (*bias*).


In [ ]:
def escalon(z):
    """Funcion de activacion escalon: 1 si z >= 0, 0 en caso contrario."""
    return 1 if z >= 0 else 0


def predecir(x, pesos, sesgo):
    """Salida del perceptron para una observacion x."""
    z = np.dot(x, pesos) + sesgo
    return escalon(z)

In [ ]:
# Grafico de la funcion escalon
zs = np.linspace(-1, 1, 400)
plt.plot(zs, [escalon(z) for z in zs], linewidth=2)
plt.title("Funcion de activacion escalon")
plt.xlabel("z (suma ponderada)")
plt.ylabel("phi(z)")
plt.grid(True)
plt.show()

## 3. La regla de aprendizaje (regla delta)

Tras cada observación se compara la salida predicha `y` con la esperada `d` y se corrige:

```
error = d - y

Δw_i = n * error * x_i      ->  w_i = w_i + Δw_i
Δθ   = n * error            ->  θ   = θ + Δθ
```

`n` es la **tasa de aprendizaje** (learning rate): regula cuánto se ajusta cada peso.

Puntos clave:

- Si `error = 0` (acierto), no hay ajuste: `Δw = 0`.
- Si `x_i = 0`, ese peso tampoco se ajusta — solo aprenden las entradas activas.
- El umbral **siempre** se ajusta cuando hay error, porque su "entrada" es constante 1.

## 4. Entrenamiento paso a paso


In [ ]:
tasa_aprendizaje = 0.045
iteraciones = 20

In [ ]:
def entrenar(entradas, etiquetas, tasa_aprendizaje, iteraciones,
             pesos=None, sesgo=None, verbose=True):
    """Entrena un perceptron con la regla delta.

    Devuelve los pesos finales, el sesgo final y el historial de errores por iteracion.
    """
    # Inicializamos de forma aleatoria los pesos y el sesgo (si no se pasan)
    pesos = np.random.rand(2) if pesos is None else np.array(pesos, dtype=float)
    sesgo = float(np.random.rand()) if sesgo is None else float(sesgo)

    historial = []

    for i in range(iteraciones):
        errores = 0

        # OJO: usar nombres distintos (x, d) para no pisar los arrays 'entradas'/'etiquetas'
        for x, d in zip(entradas, etiquetas):
            z = np.dot(x, pesos) + sesgo
            y = escalon(z)
            error = d - y

            # Actualizacion de pesos y umbral
            pesos = pesos + tasa_aprendizaje * error * x
            sesgo = sesgo + tasa_aprendizaje * error

            errores += abs(error)

            if verbose:
                print(f"  x={x} d={d} | z={z:+.3f} y={y} error={error:+d} "
                      f"| w=[{pesos[0]:.3f}, {pesos[1]:.3f}] theta={sesgo:+.3f}")

        historial.append(errores)

        if verbose:
            print(f"Iteracion {i + 1}: errores = {errores}\n")

        # Convergencia: una pasada completa sin errores
        if errores == 0:
            if verbose:
                print(f"Convergio en la iteracion {i + 1}")
            break

    return pesos, sesgo, historial

In [ ]:
pesos, sesgo, historial = entrenar(entradas, etiquetas, tasa_aprendizaje, iteraciones)

print(f"\nPesos finales: w1={pesos[0]:.3f}, w2={pesos[1]:.3f}")
print(f"Umbral final:  theta={sesgo:.3f}")

In [ ]:
# Verificacion: el perceptron reproduce la tabla de verdad?
print("x1 x2 | esperado | predicho")
for x, d in zip(entradas, etiquetas):
    print(f" {x[0]}  {x[1]} |    {d}     |    {predecir(x, pesos, sesgo)}")

In [ ]:
# Curva de convergencia: errores por iteracion
plt.plot(range(1, len(historial) + 1), historial, marker="o")
plt.title("Convergencia del perceptron (AND)")
plt.xlabel("Iteracion")
plt.ylabel("Errores en la pasada")
plt.xticks(range(1, len(historial) + 1))
plt.grid(True)
plt.show()

## 5. Reproducción del ejemplo manual de las slides

Las partes 1 a 4 de la teoría entrenan el perceptrón **a mano** partiendo de
`w1 = 0.336`, `w2 = 0.2`, `theta = 0.061` (estado al cierre de la 1.ª iteración) y `n = 0.045`,
y convergen en la 5.ª/6.ª iteración con `w1 = 0.201`, `w2 = 0.11`, `theta = -0.209`.

Corriendo el mismo punto de partida por el código deberíamos obtener esos mismos valores.


In [ ]:
pesos_slide, sesgo_slide, hist_slide = entrenar(
    entradas, etiquetas,
    tasa_aprendizaje=0.045,
    iteraciones=10,
    pesos=[0.336, 0.2],
    sesgo=0.061,
    verbose=False,
)

print(f"Codigo : w1={pesos_slide[0]:.3f}, w2={pesos_slide[1]:.3f}, theta={sesgo_slide:.3f}")
print("Slides : w1=0.201, w2=0.110, theta=-0.209")
print(f"Errores por iteracion: {hist_slide}")

> Nota: la teoría numera la iteración de partida como la 2.ª (la 1.ª ya había ocurrido antes),
> por eso el conteo de iteraciones del código arranca desplazado respecto de las slides,
> pero los pesos finales coinciden.

## 6. Visualización de la frontera de decisión

La frontera es la recta donde `z = 0`:

```
w1*x1 + w2*x2 + theta = 0   ->   x2 = -(w1*x1 + theta) / w2
```


In [ ]:
def graficar_frontera(entradas, etiquetas, pesos, sesgo, titulo):
    plt.scatter(entradas[:, 0], entradas[:, 1], c=etiquetas, cmap="cool", marker="o", s=150,
                edgecolors="black", zorder=3)

    if abs(pesos[1]) > 1e-9:
        x1 = np.linspace(-0.3, 1.3, 100)
        x2 = -(pesos[0] * x1 + sesgo) / pesos[1]
        plt.plot(x1, x2, "k--", linewidth=2, label="frontera de decision")
        plt.legend()

    plt.xlim(-0.3, 1.3)
    plt.ylim(-0.3, 1.3)
    plt.title(titulo)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.grid(True)
    plt.show()


graficar_frontera(entradas, etiquetas, pesos, sesgo, "Frontera de decision aprendida (AND)")

La recta deja el punto (1,1) — la única salida 1 — de un lado y los otros tres del otro.

## 7. La limitación: el problema del XOR

El perceptrón simple **solo** resuelve problemas linealmente separables. El caso clásico que
**no** puede resolver es el **XOR**, que devuelve 1 cuando las entradas son distintas:

| x1 | x2 | x1 XOR x2 |
|----|----|-----------|
| 0  | 0  | 0         |
| 0  | 1  | 1         |
| 1  | 0  | 1         |
| 1  | 1  | 0         |

No existe **ninguna** recta que separe {(0,1), (1,0)} de {(0,0), (1,1)}.


In [ ]:
etiquetas_xor = np.array([0, 1, 1, 0])

plt.scatter(entradas[:, 0], entradas[:, 1], c=etiquetas_xor, cmap="cool", marker="o", s=150,
            edgecolors="black")
plt.title("Compuerta logica XOR: no linealmente separable")
plt.xlabel("x1")
plt.ylabel("x2")
plt.grid(True)
plt.show()

In [ ]:
pesos_xor, sesgo_xor, hist_xor = entrenar(
    entradas, etiquetas_xor, tasa_aprendizaje=0.045, iteraciones=50, verbose=False
)

print(f"Errores por iteracion: {hist_xor}")
print(f"\nTras 50 iteraciones NO converge (los errores nunca llegan a 0).")
print("\nx1 x2 | esperado | predicho")
for x, d in zip(entradas, etiquetas_xor):
    print(f" {x[0]}  {x[1]} |    {d}     |    {predecir(x, pesos_xor, sesgo_xor)}")

Los errores oscilan sin estabilizarse: el perceptrón nunca converge con XOR.
La solución es agregar una **capa oculta** (perceptrón multicapa / MLP), que es el tema
siguiente del módulo.

## 8. Implementación con scikit-learn

Todo lo anterior está encapsulado en `sklearn.linear_model.Perceptron`, que implementa la
misma regla de aprendizaje con la API estándar `fit` / `predict` de la librería.


In [ ]:
from sklearn.linear_model import Perceptron

clf = Perceptron(max_iter=1000, eta0=0.045, random_state=42)
clf.fit(entradas, etiquetas)

print("Predicciones AND:", clf.predict(entradas))
print("Esperado        :", etiquetas)
print(f"\nPesos: {clf.coef_[0]}")
print(f"Bias : {clf.intercept_[0]}")
print(f"Accuracy: {clf.score(entradas, etiquetas):.2f}")

In [ ]:
graficar_frontera(entradas, etiquetas, clf.coef_[0], clf.intercept_[0],
                  "Frontera de decision - sklearn Perceptron (AND)")

In [ ]:
# El mismo estimador tampoco puede con XOR
clf_xor = Perceptron(max_iter=1000, eta0=0.045, random_state=42)
clf_xor.fit(entradas, etiquetas_xor)

print("Predicciones XOR:", clf_xor.predict(entradas))
print("Esperado        :", etiquetas_xor)
print(f"Accuracy: {clf_xor.score(entradas, etiquetas_xor):.2f}  <- no llega a 1.0")

## Conclusiones

- El perceptrón calcula una **suma ponderada** y le aplica una **función escalón**.
- Aprende con la **regla delta**: ajusta pesos y umbral proporcionalmente al error y a la
  tasa de aprendizaje.
- Converge en pocas iteraciones con AND, que es **linealmente separable**, y los pesos
  obtenidos por código coinciden con el cálculo manual de las slides.
- **No converge** con XOR: es su limitación fundamental, y el motivo por el que existen las
  redes multicapa.
- `sklearn.linear_model.Perceptron` implementa lo mismo con la API `fit` / `predict`.

Teoría relacionada en [`../teoria/`](../teoria/): `Perceptrón - Estructura y Fórmulas.md`,
`Compuerta Lógica AND - Parte 1` a `4`, `Limitaciones.md`,
`Implementación con Scikit-Learn.md`.
